# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
234,0.64,784.0,343.0,220.50,3.5,0.10,17.37
444,0.82,612.5,318.5,147.00,7.0,0.25,25.98
149,0.90,563.5,318.5,122.50,7.0,0.10,30.93
675,0.98,514.5,294.0,110.25,7.0,0.40,33.31
350,0.82,612.5,318.5,147.00,7.0,0.25,26.05


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

Bitkiler hassas olduğu için hataları daha fazla cezalandırması adına MSE kayıp fonksiyonunu kullanırım.

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [3]:
from sklearn.preprocessing import StandardScaler

In [5]:
X = data.drop('Average Temperature', axis=1)
y = data['Average Temperature']

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

In [8]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import cross_validate

In [9]:
model = SGDRegressor()

result = cross_validate(model, X_scaled, y, cv=10, scoring='neg_mean_squared_error')

In [10]:
result.keys()

dict_keys(['fit_time', 'score_time', 'test_score'])

In [11]:
MSE = -result['test_score'].mean()
print('MSE', MSE)

MSE 9.036571121945205


❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

In [12]:
result_1 = cross_validate(model, X_scaled, y, cv=10, scoring=['r2', 'max_error'])

In [14]:
result_1.keys()

dict_keys(['fit_time', 'score_time', 'test_r2', 'test_max_error'])

In [18]:
r2 = result_1['test_r2'].mean()
print('r2:', r2)

r2: 0.8977146242219198


In [29]:
max_error_celsius = -result_1['test_max_error'].mean()
print('max_error_celsius:', max_error_celsius)

max_error_celsius: 8.740135894115634


### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

In [19]:
model_1 = SGDRegressor(loss='epsilon_insensitive', epsilon=0)
result_2 = cross_validate(model_1, X_scaled, y, cv=10, scoring='neg_mean_absolute_error')

In [20]:
result_2.keys()

dict_keys(['fit_time', 'score_time', 'test_score'])

In [24]:
MAE = -result_2['test_score'].mean()
print('MAE:', MAE)

MAE: 2.288952319462255


❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

In [25]:
result_3 = cross_validate(model_1, X_scaled, y, cv=10, scoring=['r2', 'max_error'])

In [26]:
r2_mae = result_3['test_r2'].mean()
print('r2:', r2_mae)

r2: 0.8769368909171723


In [30]:
max_error_mae = -result_3['test_max_error'].mean()
print('max_error_mae:', max_error_mae)

max_error_mae: 10.849440352750285


## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>

MSE

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [31]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/elf/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/elf/code/elifcal/workintech-projects/W04_05-loss-functions/tests
plugins: typeguard-4.4.2, anyio-4.8.0, dash-4.1.0
collecting ... collected 3 items

test_loss_functions.py::TestLossFunctions::test_max_error_order PASSED   [ 33%]
test_loss_functions.py::TestLossFunctions::test_r2 PASSED                [ 66%]
test_loss_functions.py::TestLossFunctions::test_r2_mae PASSED            [100%]

============================== 3 passed in 0.34s ===============================


💯 You can commit your code:

git add tests/loss_functions.pickle

git commit -m 'Completed loss_functions step'

git push origin master

